# Restaurant Annual Turnover Prediction

This notebook follows the supplied sample notebook's **load → inspect → prepare → model → predict → save** flow. It adds a reliable validation score and uses all available predictors.

**Goal:** predict `Annual Turnover`. **Competition metric:** RMSE (lower is better).

## 1. Import libraries and locate the project files

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

# This works when the notebook is launched from either Hackathon/ or Hackathon/notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Train_dataset.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / 'Train_dataset.csv'
TEST_PATH = PROJECT_ROOT / 'Test_dataset.csv'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
TARGET = 'Annual Turnover'
ID_COLUMN = 'Registration Number'
RANDOM_STATE = 42

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from experiment_tracker import LOG_PATH, save_experiment

print('Project root:', PROJECT_ROOT.resolve())

## 2. Load and understand the data

In [ ]:
train = pd.read_csv(TRAIN_PATH, na_values=['NA'])
test = pd.read_csv(TEST_PATH, na_values=['NA'])

print('Training shape:', train.shape)
print('Test shape:    ', test.shape)
train.head()

In [ ]:
# The target should appear only in training data.
print('Target in training data:', TARGET in train.columns)
print('Target in test data:    ', TARGET in test.columns)

column_summary = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'missing_count': train.isna().sum(),
    'missing_percent': (train.isna().mean() * 100).round(1),
    'unique_values': train.nunique(dropna=True),
}).sort_values('missing_percent', ascending=False)
column_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train[TARGET], kde=True, ax=axes[0])
axes[0].set_title('Annual Turnover Distribution')
sns.boxplot(x=train[TARGET], ax=axes[1])
axes[1].set_title('Annual Turnover and Possible Outliers')
plt.tight_layout();

## 3. Prepare features

We do not use the registration number as a predictor because it is an identifier. We turn the opening date into useful numeric features. The supplied test file also lacks `Endorsed By`, so we explicitly align it to the training columns.

In [ ]:
def add_date_features(frame):
    result = frame.copy()
    opened = pd.to_datetime(
        result.pop('Opening Day of Restaurant'),
        dayfirst=True,
        errors='coerce',
    )
    result['Opening Year'] = opened.dt.year
    result['Opening Month'] = opened.dt.month
    result['Restaurant Age'] = 2026 - opened.dt.year
    return result

y = train[TARGET].astype(float)
X = add_date_features(train.drop(columns=[TARGET, ID_COLUMN]))
X_test = add_date_features(test.drop(columns=[ID_COLUMN]))
X_test = X_test.reindex(columns=X.columns)

print('Modeling features:', X.shape[1])
print('Columns missing from original test file:', sorted(set(X.columns) - set(add_date_features(test.drop(columns=[ID_COLUMN])).columns)))

## 4. Build the preprocessing and model pipeline

A pipeline prevents data leakage: missing-value rules and category encoding are learned separately inside each validation fold.

In [ ]:
numeric_columns = X.select_dtypes(include='number').columns.tolist()
categorical_columns = X.select_dtypes(exclude='number').columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False)),
])

preprocessing = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])

model = ExtraTreesRegressor(
    n_estimators=800,
    min_samples_leaf=2,
    max_features=0.5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', model),
])

print(f'{len(numeric_columns)} numeric and {len(categorical_columns)} categorical features')

## 5. Evaluate with cross-validation

Five-fold cross-validation trains on four parts and validates on the remaining part, repeating until every row has been held out once. This is much more trustworthy than evaluating on the same rows used for training.

In [ ]:
# CHANGE ONLY THESE VALUES FOR THE NEXT EXPERIMENT.
EXPERIMENT_PARAMETERS = {
    'n_estimators': 600,
    'min_samples_leaf': 6,
    'max_features': 0.6,
    'max_depth': 30,
}
EXPERIMENT_OBSERVATION = (
    'Testing a slightly larger leaf with square-root feature sampling and depth 35.'
)

folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
experiment_pipeline = clone(pipeline).set_params(
    model__n_estimators=EXPERIMENT_PARAMETERS['n_estimators'],
    model__min_samples_leaf=EXPERIMENT_PARAMETERS['min_samples_leaf'],
    model__max_features=EXPERIMENT_PARAMETERS['max_features'],
    model__max_depth=EXPERIMENT_PARAMETERS['max_depth'],
)
negative_mse = cross_val_score(
    experiment_pipeline,
    X,
    y,
    cv=folds,
    scoring='neg_mean_squared_error',
    n_jobs=1,
)
fold_rmse = np.sqrt(-negative_mse)
fold_results = pd.DataFrame({
    'fold': np.arange(1, 6),
    'rmse': fold_rmse,
})
display(fold_results.style.format({'rmse': '{:,.0f}'}))
print(f'Mean RMSE: {fold_rmse.mean():,.0f} (+/- {fold_rmse.std():,.0f})')

### Persistent experiment log

Each time the experiment and save cells are run, the tracker assigns the next available ID and appends the result to:

- `experiments/experiment_history.csv` — structured history for sorting and analysis.
- `experiments/EXPERIMENT_LOG.md` — readable Markdown history with observations.

Previous entries are never overwritten. Change `EXPERIMENT_PARAMETERS` and `EXPERIMENT_OBSERVATION`, then run from the validation cell through the save cell once. Running the save cell again intentionally creates another experiment ID.

## 6. Train on all rows and create the submission

After choosing a model using cross-validation, fit it on all labelled data and predict the unseen test data.

In [ ]:
experiment_pipeline.fit(X, y)
predictions = np.clip(experiment_pipeline.predict(X_test), a_min=0, a_max=None)

submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET: predictions,
})
submission.head()

In [ ]:
# Run this cell once per completed experiment. It assigns the next ID,
# saves the versioned submission, and APPENDS both experiment logs.
# Import in dependency order so this cell also works independently.
from pathlib import Path
import sys
from IPython.display import Markdown, display

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'Train_dataset.csv').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

tracker_path = str(PROJECT_ROOT / 'src')
if tracker_path not in sys.path:
    sys.path.insert(0, tracker_path)
from experiment_tracker import LOG_PATH, save_experiment

saved_experiment = save_experiment(
    model_name='ExtraTreesRegressor',
    parameters=EXPERIMENT_PARAMETERS,
    fold_rmse=fold_rmse,
    predictions=predictions,
    registration_numbers=test[ID_COLUMN],
    observation=EXPERIMENT_OBSERVATION,
)
submission = saved_experiment['submission']
print(f"Saved experiment: {saved_experiment['experiment_id']}")
print(f"Mean RMSE: {saved_experiment['mean_rmse']:,.0f}")
print('Submission:', saved_experiment['submission_path'].resolve())
print('Log:', LOG_PATH.resolve())
display(submission.head())

display(Markdown(LOG_PATH.read_text(encoding='utf-8')))

## 7. Your next experiments

Change **one thing at a time** and record the mean RMSE:

1. Compare this model with `RandomForestRegressor` or gradient boosting.
2. Try predicting `np.log1p(Annual Turnover)` and convert predictions back with `np.expm1`.
3. Explore restaurant age, cuisine counts, popularity scores, and ratings.
4. Check whether extreme turnover values dominate RMSE.
5. Keep the same cross-validation folds so comparisons remain fair.

Do not use the test set to choose a model; it has no target and represents the final unseen data.